# 🚀 Rocket Launch Delay Predictor
## Notebook 2 — Model Analysis & SHAP Explainability

> **Run `python src/train.py` before this notebook**

This notebook:
1. Loads trained models
2. Evaluates on the test set
3. Visualises SHAP feature importances
4. Explains individual predictions
5. Produces publication-ready plots

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (RocCurveDisplay, ConfusionMatrixDisplay,
                              classification_report)

plt.style.use('dark_background')
ACCENT = '#58a6ff'; DANGER = '#f85149'; SUCCESS = '#3fb950'

MODELS_DIR = '../models'

clf    = joblib.load(f'{MODELS_DIR}/xgb_classifier.pkl')
prep   = joblib.load(f'{MODELS_DIR}/clf_preprocessor.pkl')
fnames = joblib.load(f'{MODELS_DIR}/feature_names.pkl')
lgb    = joblib.load(f'{MODELS_DIR}/lgb_regressor.pkl')

with open(f'{MODELS_DIR}/training_summary.json') as f:
    summary = json.load(f)

print('Models loaded ✅')
pd.DataFrame(summary['classifier_results'])

In [ ]:
# ── Load test data ────────────────────────────────────────────────────────────
from src.data_pipeline import load_and_split
splits = load_and_split()
X_train, X_test, y_train, y_test = splits['clf']

X_test_t = prep.transform(X_test)
y_prob   = clf.predict_proba(X_test_t)[:, 1]
y_pred   = clf.predict(X_test_t)

print(classification_report(y_test, y_pred))

In [ ]:
# ── ROC Curve + Confusion Matrix ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], color=ACCENT,
                                  name='XGBoost')
axes[0].plot([0,1],[0,1],'--', color='gray', alpha=0.6)
axes[0].set_title('ROC Curve — Test Set', fontsize=12)
axes[0].set_facecolor('#161b22')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[1],
                                         colorbar=False,
                                         cmap='Blues')
axes[1].set_title('Confusion Matrix', fontsize=12)
axes[1].set_facecolor('#161b22')

plt.tight_layout()
plt.savefig('../outputs/roc_confusion.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# ── Compute SHAP values ───────────────────────────────────────────────────────
print('Computing SHAP values (this takes ~30s)...')
explainer   = shap.TreeExplainer(clf)
X_sample    = X_test_t[:600]
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# ── Top 15 features by mean |SHAP| ───────────────────────────────────────────
mean_shap = np.abs(shap_values).mean(axis=0)
top15_idx = np.argsort(mean_shap)[-15:][::-1]
top15_names = [fnames[i] for i in top15_idx]
top15_vals  = mean_shap[top15_idx]

fig, ax = plt.subplots(figsize=(9, 5))
colors = [ACCENT] * 5 + ['#8b949e'] * 10
ax.barh(range(15), top15_vals[::-1], color=colors[::-1], alpha=0.9)
ax.set_yticks(range(15))
ax.set_yticklabels(top15_names[::-1], fontsize=9)
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Top 15 Most Important Features (SHAP)', fontsize=12)
ax.set_facecolor('#161b22')
ax.grid(axis='x', alpha=0.2, color='#8b949e')
plt.tight_layout()
plt.savefig('../outputs/shap_bar_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# ── Beeswarm summary plot ────────────────────────────────────────────────────
top20_idx = np.argsort(mean_shap)[-20:]
sv20   = shap_values[:, top20_idx]
fn20   = [fnames[i] for i in top20_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor('#161b22')

for row_i, (sv_col, fname) in enumerate(zip(sv20.T, fn20)):
    jitter = np.random.normal(0, 0.1, len(sv_col))
    colors = [DANGER if v > 0 else SUCCESS for v in sv_col]
    ax.scatter(sv_col, row_i + jitter, c=colors, alpha=0.35, s=7)

ax.set_yticks(range(20))
ax.set_yticklabels(fn20, fontsize=9)
ax.axvline(0, color='#8b949e', linewidth=0.8, linestyle='--')
ax.set_xlabel('SHAP value (impact on delay probability)')
ax.set_title('SHAP Summary Plot (Beeswarm)', fontsize=12)
ax.grid(axis='x', alpha=0.12, color='#8b949e')
red_p   = mpatches.Patch(color=DANGER,  label='Increases delay risk')
green_p = mpatches.Patch(color=SUCCESS, label='Decreases delay risk')
ax.legend(handles=[red_p, green_p], facecolor='#161b22')
plt.tight_layout()
plt.savefig('../outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# ── Waterfall: Highest-risk launch ───────────────────────────────────────────
worst_idx = int(np.argmax(shap_values.sum(axis=1)))
sv_worst  = shap_values[worst_idx]
top12     = np.argsort(np.abs(sv_worst))[-12:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor('#161b22')

sv_plot = sv_worst[top12][::-1]
fn_plot = [fnames[i] for i in top12][::-1]
colors  = [DANGER if v > 0 else SUCCESS for v in sv_plot]
ax.barh(range(12), sv_plot, color=colors, alpha=0.9)

ax.set_yticks(range(12))
ax.set_yticklabels(fn_plot, fontsize=9)
ax.axvline(0, color='#8b949e', linewidth=0.8)
ax.set_xlabel('SHAP value')
ax.set_title(f'Waterfall — Highest Risk Launch (sample {worst_idx})', fontsize=12)
ax.grid(axis='x', alpha=0.12, color='#8b949e')

prob = clf.predict_proba(X_sample[worst_idx:worst_idx+1])[:, 1][0]
ax.set_xlabel(f'SHAP value  |  Predicted delay probability: {prob:.1%}')

plt.tight_layout()
plt.savefig('../outputs/waterfall_worst.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# ── SHAP Dependence: Wind Speed ───────────────────────────────────────────────
try:
    wi = fnames.index('wind_speed_kmh')
    wind_vals = X_sample[:, wi]
    wind_shap = shap_values[:, wi]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.set_facecolor('#161b22')

    sc = ax.scatter(wind_vals, wind_shap, c=wind_shap,
                    cmap='RdYlGn_r', s=8, alpha=0.5, vmin=-0.3, vmax=0.3)
    plt.colorbar(sc, ax=ax, label='SHAP value')
    ax.axhline(0, color='#8b949e', linewidth=0.8, linestyle='--')
    ax.axvline(40, color=DANGER, linewidth=1.5, linestyle=':',
               label='High-wind threshold (40 km/h)')
    ax.set_xlabel('Wind Speed (km/h)')
    ax.set_ylabel('SHAP value')
    ax.set_title('SHAP Dependence — Wind Speed', fontsize=12)
    ax.legend()
    ax.grid(alpha=0.12, color='#8b949e')

    plt.tight_layout()
    plt.savefig('../outputs/shap_dependence_wind.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
except:
    print('wind_speed_kmh not in features')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
print('=== Final Model Report ===')
print(f'Test ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}')
print()
print('Top 5 most important features:')
for i, (name, val) in enumerate(zip(top15_names[:5], top15_vals[:5]), 1):
    print(f'  {i}. {name:<35} mean|SHAP| = {val:.4f}')